# Verificación reproducible de las cifras de la tesis

**Tesis:** *Validación de una arquitectura multiagente basada en inteligencia de enjambre para la adaptación multimodal en Programación.*
**Informe contrastado:** `D2.9.7-CIERRE-BRECHAS-CORRECCION-CUADROS.docx` (versión editorial de cierre de brechas; no es la fuente de los datos).

## Qué hace este notebook
Descarga el paquete de evidencia, comprueba la **integridad** de sus archivos (SHA-256), **valida** los datos, **recalcula** las corridas desde los registros por petición, **regenera** los resultados derivados con los scripts del proyecto (sin modificarlos) y **compara** unas 870 cifras con los valores publicados en el informe.

## Alcance y límites
- Campañas con datos aquí: **N1, N2, N3, N4, N5, N8, N8b** (directorio `oe_n10`), **N9b** y **N11**. Cubre las Tablas 6, 8–11, 13, 14, 17–24, 27–29, 40 y 41. **No** se recalculan las Tablas 5/37 (estados categóricos), 15/30 (K10 histórico), 25 (antecedente), 32/44 (síntesis bibliográfica) ni lo que requiere evaluadores humanos (ver sección H).
- **N9 fue una campaña abortada y descartada** (38 de 160 iniciadas). No se usa en ningún cálculo; N9b es la réplica integrada de 160/160 corridas.
- **Todas las campañas son experimentales y no oficiales.** Publicarlas no las vuelve oficiales.
- **Reproducir una cifra no demuestra que una hipótesis sea cierta.** Solo muestra que, con estos archivos y estos scripts, el cálculo da el valor del informe.
- No hay datos de usabilidad (SUS), de expertos ni de estudiantes: 0 evaluadores reales. No se simulan.


## B. Preparación del entorno
Se instalan **solo** las dependencias fijadas (las mismas versiones que cita el informe). No se necesitan tokens, claves ni archivos privados.

In [1]:
%pip install -q numpy==2.5.4 pandas==3.0.6 scipy==1.18.1 statsmodels==0.15.0 matplotlib==3.11.2
import sys, platform, importlib
for m in ("numpy", "pandas", "scipy", "statsmodels", "matplotlib"):
    print(f"{m:12s}", importlib.import_module(m).__version__)
print("Python      ", platform.python_version())


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: <RUTA_USUARIO>\AppData\Local\Temp\rp\venv_ext\Scripts\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
numpy        2.5.4


pandas       3.0.6
scipy        1.18.1
statsmodels  0.15.0
matplotlib   3.11.2
Python       3.13.5


## C. Acceso a los datos
Por defecto el notebook **clona el repositorio público** `https://github.com/Renato5Lara/multiagent-llm-education` en la referencia `REF`.
Si ya tiene la carpeta `reproducibility/` (clonada o subida a Colab) se usa esa, o la que indique la variable de entorno `REPRO_LOCAL_ROOT`.
**Para repetir esto sin red:** copie `reproducibility/` al directorio de trabajo.

> `REF` debe existir en el remoto. Si todavía no se ha publicado esa rama, el clonado fallará con un mensaje claro: no es un error de los datos.

In [2]:
import os, subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/Renato5Lara/multiagent-llm-education.git"
REF = "repro/publicable-2026-10-19"   # rama o etiqueta con el paquete; cámbiela por la etiqueta de la versión congelada

def localizar():
    env = os.environ.get("REPRO_LOCAL_ROOT")
    if env and (Path(env) / "EVIDENCE_MANIFEST.csv").exists():
        return Path(env), f"local (REPRO_LOCAL_ROOT={env})"
    for p in (Path.cwd(), Path.cwd() / "reproducibility", Path.cwd().parent):
        if (p / "EVIDENCE_MANIFEST.csv").exists():
            return p, f"local ({p})"
    dest = Path("/content/tesis_repo") if Path("/content").exists() else Path.cwd() / "tesis_repo"
    if dest.exists():
        shutil.rmtree(dest)
    # clon parcial: solo reproducibility/ y el código del proyecto necesario (backend/adaptation_swarm); el repositorio completo es mucho mayor
    r = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", "--branch", REF, REPO_URL, str(dest)], capture_output=True, text=True)
    if r.returncode == 0:
        r = subprocess.run(["git", "-C", str(dest), "sparse-checkout", "set", "reproducibility", "backend/adaptation_swarm"], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"No se pudo clonar {REPO_URL} en la referencia {REF!r}.\n{r.stderr.strip()}\n"
                           "Compruebe que la rama/etiqueta existe en el remoto o copie reproducibility/ a mano.")
    return dest / "reproducibility", f"clonado de {REPO_URL} @ {REF}"

ROOT, origen = localizar()
print("Origen de los datos:", origen)
print("Raíz del paquete   :", ROOT)

Origen de los datos: local (<RAIZ_CLON>\reproducibility)
Raíz del paquete   : <RAIZ_CLON>\reproducibility


In [3]:
import json
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)

def run_script(name):
    r = subprocess.run([sys.executable, str(ROOT / "scripts" / name)], cwd=ROOT / "scripts", capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(r.stdout[-4000:])
    if r.stderr.strip():
        print("[stderr]", r.stderr[-1500:])
    return r.returncode

## D. Integridad de los archivos
Se recalcula el SHA-256 de **cada archivo del manifiesto** (`EVIDENCE_MANIFEST.csv`) y de los `SHA256SUMS` propios de cada campaña. Un hash prueba que el archivo es idéntico a la referencia, **no** que su contenido sea verdadero. Si algo no coincide, el análisis se detiene.

In [4]:
rc = run_script("verify_hashes.py")
rep = json.loads((ROOT / "outputs" / "hash_report.json").read_text(encoding="utf-8"))
print(pd.DataFrame(rep["sha256sums_de_campana"])[["archivo", "entradas", "coinciden", "no_publicado", "difieren"]].to_string(index=False))
if rc != 0:
    raise RuntimeError("ERROR DE INTEGRIDAD: hay archivos alterados o ausentes. Se detiene el análisis (ver outputs/hash_report.json).")
print("\nIntegridad OK:", rep["manifiesto"]["coinciden"], "de", rep["manifiesto"]["entradas"], "archivos del manifiesto coinciden.")

manifiesto: 1540/1540 coinciden; difieren=0; faltan=0
  data/experiments/oe_n10_2026-10-14_n20/SHA256SUMS: 129/131 coinciden, 1 no publicados, 1 sanitizados, 0 difieren
  data/experiments/oe_n11_2026-10-14_3N/SHA256SUMS: 368/374 coinciden, 3 no publicados, 3 sanitizados, 0 difieren
  data/experiments/oe_n1_2026-10-05/SHA256SUMS: 5/11 coinciden, 6 no publicados, 0 sanitizados, 0 difieren
  data/experiments/oe_n2_2026-10-05/SHA256SUMS: 7/10 coinciden, 2 no publicados, 1 sanitizados, 0 difieren
  data/experiments/oe_n3_2026-10-06/SHA256SUMS: 7/10 coinciden, 3 no publicados, 0 sanitizados, 0 difieren
  data/experiments/oe_n4_2026-10-06/SHA256SUMS: 10/13 coinciden, 2 no publicados, 1 sanitizados, 0 difieren
  data/experiments/oe_n5_2026-10-06/SHA256SUMS: 13/13 coinciden, 0 no publicados, 0 sanitizados, 0 difieren
  data/experiments/oe_n6_2026-10-06/SHA256SUMS: 5/5 coinciden, 0 no publicados, 0 sanitizados, 0 difieren
  data/experiments/oe_n7_2026-10-06/SHA256SUMS: 3/3 coinciden, 0 no public

## E. Validación de los datos
Comprueba columnas obligatorias, tipos, duplicados de `run_id`, valores no finitos y que el número de filas sea el planificado. **No se elimina ninguna fila.** Regla de inclusión explícita: una corrida entra al análisis solo si `status == "ok"`; las demás se informan aquí con su motivo. Se distingue dato faltante (vacío) de cero.

In [5]:
rc = run_script("validate_data.py")
val = json.loads((ROOT / "outputs" / "validation_report.json").read_text(encoding="utf-8"))
print(val["regla_de_inclusion"])
tab = pd.DataFrame([{"campaña": k, "filas": v["filas"], "planificadas": v["filas_planificadas"], "ok (aceptadas)": v["filas_ok"], "excluidas": len(v["exclusiones"]),
                     "hallazgos": len(v["hallazgos"])} for k, v in val["campanas"].items()])
display(tab)
for k, v in val["campanas"].items():
    for e in v["exclusiones"]:
        print(f"Excluida en {k}: {e['run_id']} (status = {e['status']})")
if rc != 0:
    raise RuntimeError(f"Validación fallida: {val['estado']}")

N2: 80 filas (plan 80), ok=80, exclusiones=0, hallazgos=0
N4: 60 filas (plan 60), ok=60, exclusiones=0, hallazgos=0
N8: 120 filas (plan 120), ok=120, exclusiones=0, hallazgos=0
N8b: 60 filas (plan 60), ok=60, exclusiones=0, hallazgos=0
N9b: 160 filas (plan 160), ok=160, exclusiones=0, hallazgos=0
N11: 180 filas (plan 180), ok=179, exclusiones=1, hallazgos=0
OK

Una corrida entra al análisis solo si status == 'ok' (corrida con ventana estacionaria analizable). Las demás se informan, no se borran.


,campaña,filas,planificadas,ok (aceptadas),excluidas,hallazgos
0,N2,80,80,80,0,0
1,N4,60,60,60,0,0
2,N8,120,120,120,0,0
3,N8b,60,60,60,0,0
4,N9b,160,160,160,0,0
5,N11,180,180,179,1,0


Excluida en N11: n11-oe4-3n-swarm|u100|N10-r01 (status = no_requests)


## F. Reproducción de resultados
`reproduce_results.py` comprueba tres eslabones:
1. **Peticiones → corrida.** Cada fila de `runs.csv` se recalcula con el código del proyecto (`http_window.summarize_run`) a partir de los registros por petición `raw/*.csv`.
2. **Corridas → derivados.** Se ejecutan **sin modificar** los scripts de análisis del proyecto sobre una copia temporal **en la que antes se han borrado todas las salidas**, y se comprueba que lo regenerado sea **idéntico byte a byte** a lo publicado.
3. **Derivados → tablas.** Se comparan las cifras del informe con lo recalculado.

**Ejemplo guiado, Tabla 8 (campaña N2):** las 80 corridas originales (`oe_n2_2026-10-05/campaign/runs.csv`: 2 sistemas × 4 cargas × 10 corridas) → media por sistema y carga → comparación con las 8 filas × 4 métricas del informe.

In [6]:
rc = run_script("reproduce_results.py")
ver = json.loads((ROOT / "outputs" / "verification_report.json").read_text(encoding="utf-8"))
print("Entorno:", ver["entorno"])
print("\nEslabón 1 (peticiones -> corrida):")
display(pd.DataFrame(ver["cadena_peticiones_a_corridas"])[["campana", "filas_runs_csv", "recalculadas_desde_peticiones_y_coinciden", "no_coinciden", "sin_registro_crudo", "diferencia_maxima", "estado"]])
b = pd.DataFrame(ver["derivados_byte_a_byte"])
print("\nEslabón 2 (derivados regenerados vs publicados):", b.estado.value_counts().to_dict())
display(b.head(8))

scripts del proyecto: {'oe1_verify': 0, 'oe2_reanalysis': 0, 'oe3_reanalysis': 0, 'oe4_nvar_analysis': 0, 'oe2_aux_analysis': 0, 'oe4_n11_analysis': 0, 'verif_d27': 0, 'alineacion_aptitud_f1': 0, 'reproduce_f1_v3_independent': 0}
derivados byte a byte: {'COINCIDE_BYTE_A_BYTE': 37}
cadena peticiones->corridas: [('N2', 'VERIFICADO_DESDE_DATOS'), ('N4', 'VERIFICADO_DESDE_DATOS'), ('N8', 'VERIFICADO_DESDE_DATOS'), ('N8b', 'VERIFICADO_DESDE_DATOS'), ('N9b', 'VERIFICADO_DESDE_DATOS'), ('N11', 'VERIFICADO_DESDE_DATOS'), ('N3', 'VERIFICADO_DESDE_DATOS')]
estados: {'COINCIDE': 858, 'NO_REPRODUCIBLE': 8, 'DIFERENCIA_EXPLICABLE': 3}

Entorno: {'python': '3.13.5', 'scipy': '1.18.1', 'pandas': '3.0.6', 'numpy': '2.5.4', 'statsmodels': '0.15.0'}

Eslabón 1 (peticiones -> corrida):


,campana,filas_runs_csv,recalculadas_desde_peticiones_y_coinciden,no_coinciden,sin_registro_crudo,diferencia_maxima,estado
0,N2,80,80,0,0,4.547474e-13,VERIFICADO_DESDE_DATOS
1,N4,60,60,0,0,1.818989e-12,VERIFICADO_DESDE_DATOS
2,N8,120,120,0,0,3.637979e-12,VERIFICADO_DESDE_DATOS
3,N8b,60,60,0,0,1.818989e-12,VERIFICADO_DESDE_DATOS
4,N9b,160,160,0,0,4.547474e-13,VERIFICADO_DESDE_DATOS
5,N11,180,180,0,0,1.818989e-12,VERIFICADO_DESDE_DATOS
6,N3,240,240,0,0,1.421085e-14,VERIFICADO_DESDE_DATOS



Eslabón 2 (derivados regenerados vs publicados): {'COINCIDE_BYTE_A_BYTE': 37}


,archivo,sha256_publicado,sha256_recalculado,estado
0,oe_n8_2026-10-13/oe1_verify/oe1_verify.json,19838773179daa91,19838773179daa91,COINCIDE_BYTE_A_BYTE
1,oe_n8_2026-10-13/oe2_reanalysis/oe2_contrastes.csv,56f3972a28727c0a,56f3972a28727c0a,COINCIDE_BYTE_A_BYTE
2,oe_n8_2026-10-13/oe2_reanalysis/oe2_descriptivos.csv,b774a6571873c7b7,b774a6571873c7b7,COINCIDE_BYTE_A_BYTE
3,oe_n8_2026-10-13/oe2_reanalysis/oe2_reanalysis.json,07372b571b9a5dda,07372b571b9a5dda,COINCIDE_BYTE_A_BYTE
4,oe_n8_2026-10-13/oe3_reanalysis/oe3_anova.csv,d23cdbeaaa3dde93,d23cdbeaaa3dde93,COINCIDE_BYTE_A_BYTE
5,oe_n8_2026-10-13/oe3_reanalysis/oe3_marginales.csv,cdfc349e82b45c3f,cdfc349e82b45c3f,COINCIDE_BYTE_A_BYTE
6,oe_n8_2026-10-13/oe3_reanalysis/oe3_medias_por_condicion.csv,fd81b4e13b7e2c24,fd81b4e13b7e2c24,COINCIDE_BYTE_A_BYTE
7,oe_n8_2026-10-13/oe3_reanalysis/oe3_reanalysis.json,cde73865e93b32db,cde73865e93b32db,COINCIDE_BYTE_A_BYTE


In [7]:
# Tabla 8 recalculada desde las 80 corridas de N2 (aquí, a la vista)
n2 = pd.read_csv(ROOT / "data/experiments/oe_n2_2026-10-05/campaign/runs.csv", low_memory=False)
print("Corridas en N2:", len(n2), "| status ok:", int((n2.status == "ok").sum()), "| sistemas:", sorted(n2.system.unique()), "| cargas:", sorted(n2.load.unique()))
t8 = n2[n2.status == "ok"].groupby(["system", "load"])[["p50_ms", "mean_ms", "p95_ms", "throughput_rps"]].mean().round(1)
display(t8)

Corridas en N2: 80 | status ok: 80 | sistemas: ['decision_table', 'swarm'] | cargas: [np.int64(1), np.int64(5), np.int64(10), np.int64(25)]


p50_ms  mean_ms  p95_ms  throughput_rps
system         load                                         
decision_table 1       13.3     14.7    23.9            67.4
               5       18.5     19.3    24.6           257.1
               10      33.2     35.1    42.5           283.2
               25      85.8     94.0   181.0           267.9
swarm          1       90.7     93.2   120.0            10.7
               5      284.7    298.4   436.7            16.8
               10     577.9    606.7   921.9            16.5
               25    1401.4   1523.5  2316.4            16.5

## G. Comparación con el informe
Cada fila compara un valor **publicado** (transcrito de `data/metadata/informe_D2.9.7_tablas.json`, extraído del DOCX con SHA-256 registrado) con el **recalculado**.
- Tolerancia: media unidad del último decimal publicado (redondeo). `COINCIDE` ≤ ½ unidad; `DIFERENCIA_EXPLICABLE` ≤ 1 unidad; `DISCREPANCIA` > 1 unidad.
- `NO_REPRODUCIBLE` indica que falta archivo, variable o procedimiento, y dice cuál.

In [8]:
comp = pd.DataFrame(ver["comprobaciones"])
print(ver["conteo_estados"])
resumen = comp.groupby(["tabla", "estado"]).size().unstack(fill_value=0)
display(resumen)
cols = ["id", "publicado", "recalculado", "dif_abs", "dif_rel", "tolerancia", "estado", "fuente_evidencia"]
display(comp[comp.estado != "NO_REPRODUCIBLE"][cols].head(12))
print("\nNo reproducibles y motivo:")
display(comp[comp.estado == "NO_REPRODUCIBLE"][["id", "campana", "motivo"]])

{'COINCIDE': 858, 'NO_REPRODUCIBLE': 8, 'DIFERENCIA_EXPLICABLE': 3}


estado,COINCIDE,DIFERENCIA_EXPLICABLE,NO_REPRODUCIBLE
tabla,,,
Tabla 10,24,0,0
Tabla 11,96,0,0
Tabla 13,36,3,0
Tabla 14,36,0,0
Tabla 17,36,0,0
Tabla 18,36,0,0
Tabla 19,60,0,0
Tabla 20,18,0,0
Tabla 21,72,0,0


,id,publicado,recalculado,dif_abs,dif_rel,tolerancia,estado,fuente_evidencia
0,T8-decision_table-u1-p50_ms,"13,3",13.31595,0.015950,0.001199,0.05,COINCIDE,oe_n2_2026-10-05/campaign/runs.csv (80 corridas)
1,T8-decision_table-u1-mean_ms,"14,7",14.652611,-0.047389,-0.003224,0.05,COINCIDE,oe_n2_2026-10-05/campaign/runs.csv (80 corridas)
2,T8-decision_table-u1-p95_ms,"23,9",23.93858,0.038580,0.001614,0.05,COINCIDE,oe_n2_2026-10-05/campaign/runs.csv (80 corridas)
3,T8-decision_table-u1-throughput_rps,"67,4",67.394385,-0.005615,-0.000083,0.05,COINCIDE,oe_n2_2026-10-05/campaign/runs.csv (80 corridas)
4,T8-swarm-u1-p50_ms,"90,7",90.72365,0.023650,0.000261,0.05,COINCIDE,oe_n2_2026-10-05/campaign/runs.csv (80 corridas)
5,T8-swarm-u1-mean_ms,"93,2",93.247435,0.047435,0.000509,0.05,COINCIDE,oe_n2_2026-10-05/campaign/runs.csv (80 corridas)
6,T8-swarm-u1-p95_ms,"120,0",119.98372,-0.016280,-0.000136,0.05,COINCIDE,oe_n2_2026-10-05/campaign/runs.csv (80 corridas)
7,T8-swarm-u1-throughput_rps,"10,7",10.713206,0.013206,0.001234,0.05,COINCIDE,oe_n2_2026-10-05/campaign/runs.csv (80 corridas)
8,T8-decision_table-u5-p50_ms,"18,5",18.5096,0.009600,0.000519,0.05,COINCIDE,oe_n2_2026-10-05/campaign/runs.csv (80 corridas)
9,T8-decision_table-u5-mean_ms,"19,3",19.255632,-0.044368,-0.002299,0.05,COINCIDE,oe_n2_2026-10-05/campaign/runs.csv (80 corridas)



No reproducibles y motivo:


,id,campana,motivo
677,T6-Usuarios (carga) fre,N1,variable no incluida en oe1_verify.json (cálculo no localizado)
688,T27-perfil,N5,"la mediana, desviación y rango por perfil no se recalculan en esta entrega"
689,T27-sus,N5,no existen datos: 0 evaluadores reales; no se simulan
864,PEND-Tablas 5 y 37,N1,estados MET/NOT_MET: texto categórico del informe de N1 (oe1_report.json); no hay cálc...
865,PEND-Tablas 15 y 30,K10 (v2 histórico),resultados históricos de la especificación v2 (K10) en backend/official_runs; no recal...
866,PEND-Tabla 25,antecedente,"pruebas exploratorias de carga (backend/loadtest/results, ya en Git); no recalculadas ..."
867,PEND-Tablas 32 y 44,transversal,síntesis bibliográfica: depende de consultas externas (Crossref/OpenAlex); no verifica...
868,PEND-Tabla 36,N5,instrumento SUS sin aplicar: no hay datos; no se simulan


## H. Limitaciones y conclusión

In [9]:
c = ver["conteo_estados"]
tr = pd.read_csv(ROOT / "RESULTS_TRACEABILITY.csv")
print("Cifras por estado:", ver["conteo_estado_cifra"])
print(f"  (de ellas, {c.get('DIFERENCIA_EXPLICABLE', 0)} difieren una unidad del último decimal por doble redondeo del informe)")
print("Tablas del informe por estado:", tr.estado.value_counts().to_dict())
display(tr[["tabla_informe", "campana", "estado", "cifras_comprobadas", "verificadas_desde_datos", "no_reproducibles", "observacion"]])

Cifras por estado: {'VERIFICADO_DESDE_DATOS': 861, 'NO_REPRODUCIBLE': 8}
  (de ellas, 3 difieren una unidad del último decimal por doble redondeo del informe)
Tablas del informe por estado: {'VERIFICADO_DESDE_DATOS': 18, 'PARCIAL': 6, 'NO_APLICA': 2, 'NO_REPRODUCIBLE': 1}


,tabla_informe,campana,estado,cifras_comprobadas,verificadas_desde_datos,no_reproducibles,observacion
0,Tabla 6,N1,PARCIAL,17,16,1,la fila «Usuarios frente al propio puntaje» no está en oe1_verify.json
1,Tabla 8,N2,VERIFICADO_DESDE_DATOS,32,32,0,recalculado desde datos publicados; coincide a la precisión publicada
2,Tabla 9,N2,VERIFICADO_DESDE_DATOS,32,32,0,recalculado desde datos publicados; coincide a la precisión publicada
3,Tabla 10,N2,VERIFICADO_DESDE_DATOS,24,24,0,recalculado desde datos publicados; coincide a la precisión publicada
4,Tabla 11,N9b,VERIFICADO_DESDE_DATOS,96,96,0,N9 (abortada) no interviene
5,Tabla 13,N3,VERIFICADO_DESDE_DATOS,39,39,0,η² y 2 medias difieren 1 unidad del último decimal por doble redondeo del informe
6,Tabla 14,N3,VERIFICADO_DESDE_DATOS,36,36,0,recalculado desde datos publicados; coincide a la precisión publicada
7,Tabla 17,N4,VERIFICADO_DESDE_DATOS,36,36,0,recalculado desde datos publicados; coincide a la precisión publicada
8,Tabla 18,N8,VERIFICADO_DESDE_DATOS,36,36,0,recalculado desde datos publicados; coincide a la precisión publicada
9,Tabla 19,N8,VERIFICADO_DESDE_DATOS,60,60,0,"«Mayor p de Holm» entre P95, throughput y T_conv"


**Qué sí y qué no demuestra esta ejecución**
- Con estos archivos y estos scripts, las cifras de las tablas marcadas `VERIFICADO_DESDE_DATOS` **se recalculan** a la precisión publicada desde `runs.csv`, `raw_observations.csv` o las réplicas guardadas, y para N2, N3, N4, N8, N8b, N9b y N11 también se comprobó que `runs.csv` sale de los registros por petición. Lo que queda sin comprobar está en la tabla anterior (`PARCIAL`, `NO_REPRODUCIBLE`): estados categóricos de N1, K10 histórico, pruebas de carga exploratorias, síntesis bibliográfica, SUS y expertos.
- La reproducibilidad computacional **no equivale a autenticidad experimental**: un hash acredita la identidad de un archivo frente a una referencia, no que su contenido sea verdadero. Que los registros por petición sean reales (y no fabricados) no se puede demostrar con este cómputo: solo que las métricas salen de ellos.
- Los resultados son **experimentales y no oficiales**; los de N5 usan la especificación v3 provisional del F1; los de K10 son históricos (v2).
- La métrica de usabilidad y la validación con expertos requieren evaluadores humanos y **no existen**; no se simulan.
- Esta ejecución no prueba causalidad ni confirma hipótesis.

## I. Exportación
Se guardan el informe de verificación y las tablas en una carpeta **nueva** con marca de tiempo; los archivos originales del paquete no se sobrescriben.

In [10]:
import datetime, zipfile
dest = Path("verificacion_" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S"))
(dest).mkdir()
for f in [ROOT / "outputs" / n for n in ("verification_report.json", "hash_report.json", "validation_report.json")] + list((ROOT / "outputs" / "tables").glob("*.csv")):
    shutil.copy2(f, dest / f.name)
zpath = shutil.make_archive(str(dest), "zip", dest)
print("Exportado en:", dest.resolve(), "\nZIP:", zpath)
try:
    from google.colab import files
    files.download(zpath)
except ImportError:
    pass

Exportado en: <RAIZ_CLON>\reproducibility\notebooks\verificacion_20261019_005423 
ZIP: <RAIZ_CLON>\reproducibility\notebooks\verificacion_20261019_005423.zip
